In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

import xgboost as xgb

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    classification_report)

In [2]:
df_train = pd.read_parquet("dataset/df_train_clean.parquet")
df_test = pd.read_parquet("dataset/df_test_clean.parquet")

In [3]:
# =========================
# 1. Split
# =========================

X = df_train.drop(columns=["Will_Buy_EV"])
y = df_train["Will_Buy_EV"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [4]:
model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)

In [5]:
# =========================
# 4. Grid Search
# =========================

param_grid = {
    "n_estimators": [400, 500, 600],
    "max_depth": [2, 3, 5],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=20,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    verbose=2,
    random_state=42
)



In [6]:
random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)
print("Best AUC score:", random_search.best_score_)

best_model = random_search.best_estimator_

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best params: {'subsample': 1.0, 'n_estimators': 600, 'max_depth': 2, 'learning_rate': 0.2, 'colsample_bytree': 1.0}
Best AUC score: 0.9418492337454024


In [7]:
# =========================
# 5. Probabilités
# =========================


proba = best_model.predict_proba(X_test)[:, 1]

In [8]:
logloss = log_loss(y_test, proba)
print("LogLoss :", logloss)

LogLoss : 0.22640617191791534


In [9]:
roc_auc = roc_auc_score(y_test, proba)

print(f"ROC-AUC: {roc_auc:.4f}")

ROC-AUC: 0.9418
